# 💹 Quantum Portfolio Optimisation
### Quantum for Finance — Quantum for Humanity

This notebook demonstrates quantum portfolio optimisation using **QAOA (Quantum Approximate Optimisation Algorithm)** on IBM Quantum hardware via Qiskit.

**Learning source:** [IBM Quantum Learning — Finance](https://learning.quantum.ibm.com)

---

## Background

The **Markowitz Mean-Variance Portfolio Optimisation** problem selects a portfolio of $n$ assets to maximise expected return while minimising risk (variance):

$$\min_{x \in \{0,1\}^n} \; q \cdot x^T \Sigma x - \mu^T x$$

where:
- $x_i = 1$ if asset $i$ is included, $0$ otherwise
- $\Sigma$ = covariance matrix of asset returns
- $\mu$ = expected return vector
- $q$ = risk appetite parameter

This is an **NP-hard combinatorial optimisation** problem — a natural candidate for QAOA.

In [ ]:
# Install required packages (run once)
# !pip install qiskit qiskit-finance qiskit-optimization qiskit-algorithms

import numpy as np
import matplotlib.pyplot as plt
from qiskit_finance.applications.optimization import PortfolioOptimization
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Sampler

print('✅ Imports successful')

## Step 1: Define the Portfolio Problem

In [ ]:
# --- Define assets (e.g., 4 stocks) ---
num_assets = 4
seed = 123

# Simulate expected returns and covariance matrix
np.random.seed(seed)
mu = np.array([0.12, 0.10, 0.15, 0.08])   # expected annual returns
sigma = np.array([                          # covariance matrix
    [0.10, 0.02, 0.01, 0.03],
    [0.02, 0.08, 0.02, 0.01],
    [0.01, 0.02, 0.12, 0.02],
    [0.03, 0.01, 0.02, 0.06]
])

# Assets: AAPL, MSFT, GOOGL, AMZN (illustrative)
assets = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
risk_factor = 0.5  # q: risk appetite
budget = 2         # invest in exactly 2 assets

print(f'Assets: {assets}')
print(f'Expected returns: {mu}')
print(f'Risk factor (q): {risk_factor}')
print(f'Budget (number of assets to select): {budget}')

In [ ]:
# --- Build the Qiskit Finance Portfolio Optimisation problem ---
portfolio = PortfolioOptimization(
    expected_returns=mu,
    covariances=sigma,
    risk_factor=risk_factor,
    budget=budget
)
qp = portfolio.to_quadratic_program()
print(qp.export_as_lp_string())

## Step 2: Classical Baseline (NumPy Exact Solver)

In [ ]:
# Solve classically for comparison
exact_solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact_solver.solve(qp)

print('Classical Optimal Portfolio:')
for i, x in enumerate(exact_result.x):
    if x > 0.5:
        print(f'  ✅ Include {assets[i]} (weight={x:.0f})')
print(f'Objective value: {exact_result.fval:.4f}')

## Step 3: Quantum Solution with QAOA

In [ ]:
# --- Run QAOA on simulator ---
sampler = Sampler()
optimizer = COBYLA(maxiter=300)

qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=2)
qaoa_optimizer = MinimumEigenOptimizer(qaoa)

qaoa_result = qaoa_optimizer.solve(qp)

print('\nQAOA Quantum Portfolio:')
for i, x in enumerate(qaoa_result.x):
    if x > 0.5:
        print(f'  ⚛️  Include {assets[i]} (weight={x:.0f})')
print(f'Objective value: {qaoa_result.fval:.4f}')

## Step 4: Visualise the Efficient Frontier

In [ ]:
# Plot expected return vs risk for all asset combinations
from itertools import combinations

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#8B5CF6', '#FF6B9D', '#FF9933', '#6EA8FE', '#138808']

for k in range(1, num_assets + 1):
    for combo in combinations(range(num_assets), k):
        x = np.zeros(num_assets)
        x[list(combo)] = 1
        ret = mu @ x
        risk = x @ sigma @ x
        label = '+'.join([assets[i] for i in combo])
        ax.scatter(risk, ret, s=80, alpha=0.7, color=colors[k % len(colors)])
        ax.annotate(label, (risk, ret), fontsize=7, ha='left', va='bottom')

# Mark QAOA solution
x_q = qaoa_result.x
ret_q = mu @ x_q
risk_q = x_q @ sigma @ x_q
ax.scatter(risk_q, ret_q, s=200, color='red', zorder=5, label='QAOA Solution', marker='*')

ax.set_xlabel('Portfolio Risk (Variance)', fontsize=12)
ax.set_ylabel('Expected Return', fontsize=12)
ax.set_title('Quantum Portfolio Optimisation — Efficient Frontier', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Run on Real IBM Quantum Hardware (Optional)

To run on a real quantum computer, replace the `Sampler()` with an IBM Quantum Runtime Sampler:

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService(channel='ibm_quantum', token='YOUR_TOKEN')
backend = service.least_busy(operational=True, simulator=False)
sampler = Sampler(mode=backend)
```

Learn more at [IBM Quantum Learning](https://learning.quantum.ibm.com)

---

## 🌍 Quantum for Humanity — Finance Application

This technique can be directly applied to **microfinance portfolio optimisation**:
- Optimise loan disbursement across underserved communities
- Minimise default risk while maximising social impact
- Scale to millions of micro-borrowers using quantum speedup

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*